In [1]:
!pip install -q transformers==4.46.0 \
               peft==0.12.0 \
               trl==0.9.4 \
               bitsandbytes \
               datasets \
               sentencepiece \
               accelerate \
               lxml \
               requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.4 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.7 MB/s eta 0:00:00


In [2]:
import os
import re
import random
import requests
from lxml import html
from datasets import Dataset
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments

In [3]:
os.makedirs("pgns", exist_ok=True)

url = "https://lichess.org/api/games/user/DrNykterstein?max=300&pgnInJson=false"

print("Downloading small PGN sample from Lichess...")
pgn_text = requests.get(url).text

with open("pgns/sample_games.pgn", "w") as f:
    f.write(pgn_text)

print("Saved sample PGN with ~300 games.")

Saved sample PGN with ~300 games.


In [4]:
def extract_games(filepath):
    games = []
    current = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("["):
                continue
            if line.strip() == "":
                if current:
                    games.append(" ".join(current))
                    current = []
                continue

            moves = re.findall(r"\d+\.\s*([^\s]+)(?:\s+([^\s]+))?", line)
            for w, b in moves:
                current.append(w)
                if b not in [None, ""]:
                    current.append(b)
    return games

games = extract_games("pgns/sample_games.pgn")
len(games)

300

In [5]:
def make_comment(move):
    templates = [
        f"{move} helps control the center.",
        f"{move} develops a key piece.",
        f"{move} puts pressure on the opponent.",
        f"{move} prepares for kingside safety.",
        f"{move} opens important tactical ideas.",
        f"{move} challenges the opponent's structure."
    ]
    return random.choice(templates)

data = []

for game in games:
    moves = game.split()
    for mv in moves:
        data.append({
            "instruction": "Explain this chess move.",
            "input": mv,
            "output": make_comment(mv)
        })

len(data)

25718

In [6]:
dataset = Dataset.from_list(data)
dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 25718
})

In [7]:
from huggingface_hub import login

login()  # will ask for your HF token

In [8]:
model_name = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True,
)

peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_cfg)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

trainable params: 3,194,880 || all params: 2,617,536,768 || trainable%: 0.1221


In [9]:
def format_example(ex):
    text = (
        f"<start_of_turn>user\nExplain the move: {ex['input']}"
        f"<end_of_turn>\n<start_of_turn>assistant\n{ex['output']}<end_of_turn>"
    )
    return {"text": text}

train_dataset = dataset.map(format_example)

Map:   0%|          | 0/25718 [00:00<?, ? examples/s]

In [10]:
training_args = TrainingArguments(
    output_dir="gemma-2-chess",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=10,
    warmup_steps=20,
    max_steps=300,
    learning_rate=2e-4,
    fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:2041: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:269: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:307: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override

Map:   0%|          | 0/25718 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:397: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:402: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
max_steps is given, it will override any value given in num_train_epochs


In [11]:
trainer.train()

model.save_pretrained("gemma-2-chess-lora")
tokenizer.save_pretrained("gemma-2-chess-lora")

Step,Training Loss
10,9.487100
20,6.436400
30,2.836300
40,1.466800
50,1.049100
60,0.822000
70,0.714700
80,0.599700
90,0.506300
100,0.439500


('gemma-2-chess-lora/tokenizer_config.json',
 'gemma-2-chess-lora/special_tokens_map.json',
 'gemma-2-chess-lora/tokenizer.model',
 'gemma-2-chess-lora/added_tokens.json',
 'gemma-2-chess-lora/tokenizer.json')

In [12]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

prompt = """<start_of_turn>user
Explain the move: Nc3<end_of_turn>
<start_of_turn>assistant"""

out = pipe(prompt, max_new_tokens=75)
print(out[0]["generated_text"])

The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'GraniteForCausalLM', 'GraniteMoeForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'Mamba2ForCausalLM', 'MarianForCausalLM', 'MBartForCausa

<start_of_turn>user
Explain the move: Nc3<end_of_turn>
<start_of_turn>assistant
Nc3 opens important tactical ideas.


In [17]:
def generate_clean(move="Nc3"):
    prompt = (
        f"<start_of_turn>user\n"
        f"Explain the chess move {move} in at least two sentences."
        f"<end_of_turn>\n"
        f"<start_of_turn>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1
    )

    decoded = tokenizer.decode(output[0], skip_special_tokens=False)

    # Extract only assistant reply after the last assistant tag
    if "<start_of_turn>assistant" in decoded:
        decoded = decoded.split("<start_of_turn>assistant")[-1]

    # Remove any leftover tokens
    decoded = decoded.replace("<end_of_turn>", "").strip()

    return decoded

In [19]:
result = generate_clean("Nc3")
print(result)

Nc3 helps control the center.


In [20]:
reference = """
Nc3 develops the knight toward the center and improves White’s piece activity.
It also supports central control and prepares future possibilities like d4 or Bb5.
""".strip()

generated = result.strip()
print("Generated Output:\n", generated)

Generated Output:
 Nc3 helps control the center.


In [21]:
def generate_multi(move="Nc3"):
    prompt = (
        f"<start_of_turn>user\n"
        f"Explain the chess move {move}. Your answer MUST contain at least 2–3 full sentences and cannot be shorter than 40 words."
        f"<end_of_turn>\n"
        f"<start_of_turn>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        temperature=0.8,
        top_p=0.95,
        do_sample=True,
        min_length=60,          # ⬅ force longer output
        repetition_penalty=1.1,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Extract only assistant content
    decoded = decoded.split("<start_of_turn>assistant")[-1]
    decoded = decoded.replace("<end_of_turn>", "").strip()

    return decoded

In [22]:
result = generate_multi("Nc3")
print(result)

Nc3 opens important tactical ideas. It puts pressure on the opponent. 
bxc3+ develops a key piece.


In [15]:
!pip install -q nltk rouge-score bert-score
import nltk
nltk.download('punkt')

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [16]:
# BLEU
smooth = SmoothingFunction().method1
bleu = sentence_bleu(
    [reference.split()],
    generated.split(),
    smoothing_function=smooth
)

# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)
rouge = scorer.score(reference, generated)

# BERTScore
P, R, F1 = bertscore([generated], [reference], lang="en")

print("=== Evaluation Metrics ===")
print(f"BLEU Score      : {bleu:.4f}")
print(f"ROUGE-1         : {rouge['rouge1'].fmeasure:.4f}")
print(f"ROUGE-L         : {rouge['rougeL'].fmeasure:.4f}")
print(f"BERTScore (F1)  : {F1[0]:.4f}")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


=== Evaluation Metrics ===
BLEU Score      : 0.0096
ROUGE-1         : 0.1364
ROUGE-L         : 0.0909
BERTScore (F1)  : 0.8516
